# Passo 4 – Treino, avaliação e testes

Este notebook faz a parte principal da pesquisa.

1. **Treino e ajuste** de Naive Bayes, SVM linear e XGBoost no Fake.Br (somente a parte de treino), com **validação cruzada em grupos** (os pares de notícias nunca ficam separados).
2. Comparação **com e sem stemming**.
3. **Teste interno**: parte de teste do Fake.Br.
4. **Teste de significância**: McNemar entre os pares de algoritmos, com correção de Bonferroni.
5. **Teste externo eleitoral**: boatos da FactPolCheckBr. Só há textos falsos, então a medida é a **taxa de acerto nos falsos (recall)**, lida junto com a **taxa de falsos positivos** medida no teste interno.
6. **Experimento de viés**: treino na FakeRecogna e teste na FactPolCheckBr com o **texto da checagem** (escrito pela agência) e com o **boato extraído**.
7. **Interpretação**: palavras mais pesadas e erros.

**Pré-requisito:** notebooks 00 a 03 executados.

**Tempo:** a busca de parâmetros pode levar de 10 a 40 minutos no Colab. Não feche a aba.

In [ ]:
import os, re, unicodedata, json, time, warnings, platform, subprocess
import numpy as np
import pandas as pd
import sklearn, xgboost, scipy, statsmodels
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score, roc_auc_score, confusion_matrix
from xgboost import XGBClassifier
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.proportion import proportion_confint
warnings.filterwarnings('ignore')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA_TCC = '/content/drive/MyDrive/TCC'
    BRUTOS = '/content/dados_brutos'
except ImportError:
    PASTA_TCC = os.path.abspath('TCC')
    BRUTOS = os.path.join(PASTA_TCC, 'dados', 'brutos')

PROC = os.path.join(PASTA_TCC, 'dados', 'processados')
RES = os.path.join(PASTA_TCC, 'resultados')
SEED = 42
N_FOLDS = 5
np.random.seed(SEED)

versoes = {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
           'scikit-learn': sklearn.__version__, 'xgboost': xgboost.__version__,
           'scipy': scipy.__version__, 'statsmodels': statsmodels.__version__}
with open(os.path.join(RES, 'versoes_bibliotecas.txt'), 'w') as f:
    for k, v in versoes.items():
        f.write(f'{k}=={v}\n')
print(versoes)

In [ ]:
fake = pd.read_csv(os.path.join(PROC, 'fakebr_prep.csv'))
reco = pd.read_csv(os.path.join(PROC, 'recogna_prep.csv'))
fpc = pd.read_csv(os.path.join(PROC, 'fpc_prep.csv'))
for d in (fake, reco, fpc):
    d[['texto_proc', 'texto_stem']] = d[['texto_proc', 'texto_stem']].fillna('')
for d in (fake, reco):
    d['rotulo'] = d['rotulo'].astype(int)

tr = fake[fake.particao == 'treino'].reset_index(drop=True)
te = fake[fake.particao == 'teste'].reset_index(drop=True)
reco_tr = reco[reco.particao == 'treino'].reset_index(drop=True)
reco_te = reco[reco.particao == 'teste'].reset_index(drop=True)
print('Fake.Br treino/teste:', len(tr), len(te), '| FakeRecogna treino/teste:', len(reco_tr), len(reco_te), '| FactPolCheckBr:', len(fpc))
assert set(tr.par_id).isdisjoint(set(te.par_id))

## 1. Modelos e parâmetros testados

Todos usam **TF-IDF** como representação. O TF-IDF é calculado **dentro de cada rodada da validação cruzada** (via `Pipeline`), para que o teste de cada rodada não influencie o vocabulário.

| Algoritmo | Parâmetros testados |
|---|---|
| Naive Bayes (multinomial) | n-gramas (1,1) ou (1,2); alpha 0,1 / 0,5 / 1,0 |
| SVM linear (`LinearSVC`) | n-gramas (1,1) ou (1,2); C 0,1 / 1 / 10 |
| XGBoost | só palavras soltas, 10 mil atributos; árvores 200 ou 400; profundidade 4 ou 6; taxa de aprendizado 0,1 |

A métrica da busca é o **F1 da classe falsa**, em 5 partições que **não separam pares** (`StratifiedGroupKFold`).

**Escolha de projeto a declarar:** usamos SVM **linear** (não RBF) porque é o padrão para texto com TF-IDF, é bem mais rápido e costuma render igual em dados esparsos.

In [ ]:
def pipe_nb():
    p = Pipeline([('tfidf', TfidfVectorizer(min_df=2, max_features=50000, sublinear_tf=True)), ('clf', MultinomialNB())])
    return p, {'tfidf__ngram_range': [(1, 1), (1, 2)], 'clf__alpha': [0.1, 0.5, 1.0]}

def pipe_svm():
    p = Pipeline([('tfidf', TfidfVectorizer(min_df=2, max_features=50000, sublinear_tf=True)), ('clf', LinearSVC(random_state=SEED, max_iter=5000))])
    return p, {'tfidf__ngram_range': [(1, 1), (1, 2)], 'clf__C': [0.1, 1, 10]}

def pipe_xgb():
    p = Pipeline([('tfidf', TfidfVectorizer(min_df=2, max_features=10000, sublinear_tf=True)),
                  ('clf', XGBClassifier(tree_method='hist', n_jobs=1, random_state=SEED, eval_metric='logloss', learning_rate=0.1))])
    return p, {'clf__n_estimators': [200, 400], 'clf__max_depth': [4, 6]}

CONFIG = {'Naive Bayes': pipe_nb, 'SVM': pipe_svm, 'XGBoost': pipe_xgb}
COLUNA = {'sem_stem': 'texto_proc', 'com_stem': 'texto_stem'}

## 2. Busca de parâmetros com validação cruzada em grupos

In [ ]:
buscas = {}
linhas_cv = []
for modelo, fabrica in CONFIG.items():
    for variante, col in COLUNA.items():
        t0 = time.time()
        pipe, grade = fabrica()
        cv = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
        gs = GridSearchCV(pipe, grade, cv=cv, scoring='f1', n_jobs=-1, refit=True)
        gs.fit(tr[col], tr['rotulo'], groups=tr['par_id'])
        buscas[(modelo, variante)] = gs
        i = gs.best_index_
        linhas_cv.append({'modelo': modelo, 'texto': variante, 'f1_cv_media': round(gs.best_score_, 4),
                          'f1_cv_desvio': round(gs.cv_results_['std_test_score'][i], 4),
                          'melhores_parametros': str(gs.best_params_), 'minutos': round((time.time() - t0) / 60, 1)})
        print(linhas_cv[-1])
quadro_cv = pd.DataFrame(linhas_cv)
quadro_cv

## 3. Teste interno (Fake.Br, parte de teste)

Esta é a avaliação **principal**. O conjunto de teste nunca foi usado para escolher parâmetros.

In [ ]:
def pontuacao(modelo, X):
    return modelo.decision_function(X) if hasattr(modelo, 'decision_function') else modelo.predict_proba(X)[:, 1]

linhas_te, pred_te = [], {}
for (modelo, variante), gs in buscas.items():
    col = COLUNA[variante]
    y_pred = gs.predict(te[col])
    pred_te[(modelo, variante)] = y_pred
    p, r, f, _ = precision_recall_fscore_support(te['rotulo'], y_pred, average='binary', pos_label=1)
    tn, fp, fn, tp = confusion_matrix(te['rotulo'], y_pred).ravel()
    linhas_te.append({'modelo': modelo, 'texto': variante, 'acuracia': accuracy_score(te['rotulo'], y_pred),
                      'precisao_falso': p, 'recall_falso': r, 'f1_falso': f,
                      'f1_macro': f1_score(te['rotulo'], y_pred, average='macro'),
                      'auc': roc_auc_score(te['rotulo'], pontuacao(gs, te[col])),
                      'taxa_falso_positivo': fp / (fp + tn), 'VP': tp, 'FP': fp, 'FN': fn, 'VN': tn})
quadro_te = pd.DataFrame(linhas_te).round(4)
quadro_te

In [ ]:
# Escolha da versão do texto de cada algoritmo: a de maior F1 na VALIDAÇÃO CRUZADA (nunca pelo teste)
escolhido = {}
for modelo in CONFIG:
    sub = quadro_cv[quadro_cv.modelo == modelo].sort_values('f1_cv_media', ascending=False)
    escolhido[modelo] = sub.iloc[0]['texto']
print('Versão escolhida por validação cruzada:', escolhido)

# Efeito do stemming no teste (para a discussão)
efeito = quadro_te.pivot(index='modelo', columns='texto', values='f1_falso')
efeito['diferenca_com_menos_sem'] = efeito['com_stem'] - efeito['sem_stem']
efeito.round(4)

## 4. Teste de McNemar (comparação entre algoritmos)

O McNemar compara **dois algoritmos no mesmo conjunto de teste** olhando só os casos em que um acerta e o outro erra. São 3 comparações, então o nível de significância cai para **0,05 / 3 ≈ 0,0167** (Bonferroni).

In [ ]:
import itertools
modelos = list(CONFIG)
linhas_mc = []
for a, b in itertools.combinations(modelos, 2):
    ya, yb = pred_te[(a, escolhido[a])], pred_te[(b, escolhido[b])]
    ok_a, ok_b = (ya == te['rotulo'].values), (yb == te['rotulo'].values)
    n10, n01 = int((ok_a & ~ok_b).sum()), int((~ok_a & ok_b).sum())
    res = mcnemar([[int((ok_a & ok_b).sum()), n10], [n01, int((~ok_a & ~ok_b).sum())]], exact=True)
    linhas_mc.append({'A': a, 'B': b, 'so_A_acerta': n10, 'so_B_acerta': n01, 'p_valor': res.pvalue,
                      'significativo_bonferroni': res.pvalue < 0.05 / 3})
quadro_mc = pd.DataFrame(linhas_mc)
quadro_mc

## 5. Teste externo eleitoral (FactPolCheckBr)

**Como ler (importante):**

- O conjunto eleitoral só tem textos **falsos**. Então só dá para medir o **recall**: de todos os boatos, quantos o modelo marcou como falso.
- Um modelo que marca **tudo** como falso teria recall de 100%. Por isso o recall é lido **junto com a taxa de falsos positivos do teste interno** (quantas notícias verdadeiras do Fake.Br foram marcadas como falsas).
- O intervalo de confiança é o de **Wilson, 95%**.
- A diferença de formato (boatos curtos × notícias longas) é uma limitação e deve ser declarada.

In [ ]:
def recall_fpc(gs, variante, dados):
    pred = gs.predict(dados[COLUNA[variante]])
    k, n = int(pred.sum()), len(pred)
    lo, hi = proportion_confint(k, n, method='wilson')
    return pred, k / n, lo, hi

linhas_ext, pred_fpc = [], {}
for (modelo, variante), gs in buscas.items():
    pred, rec, lo, hi = recall_fpc(gs, variante, fpc)
    pred_fpc[(modelo, variante)] = pred
    fpr = quadro_te[(quadro_te.modelo == modelo) & (quadro_te.texto == variante)]['taxa_falso_positivo'].iloc[0]
    linhas_ext.append({'modelo': modelo, 'texto': variante, 'recall_boatos': round(rec, 4),
                       'ic95_inf': round(lo, 4), 'ic95_sup': round(hi, 4),
                       'taxa_falso_positivo_fakebr': fpr})
quadro_ext = pd.DataFrame(linhas_ext)
quadro_ext

In [ ]:
# Recall por agência de checagem (o formato do boato muda de agência para agência)
fpc_a = fpc.copy()
for modelo in CONFIG:
    fpc_a['acerto_' + modelo] = pred_fpc[(modelo, escolhido[modelo])]
por_agencia = fpc_a.groupby('agencia').agg(n=('id', 'size'), **{m: ('acerto_' + m, 'mean') for m in CONFIG}).round(3)
por_agencia

## 6. Experimento de viés: texto da agência × boato extraído

**Pergunta:** o modelo aprende a linguagem típica de **texto de agência de checagem** (e não a linguagem da desinformação)?

**Como:** treinamos na FakeRecogna (textos feitos a partir de agências) e no Fake.Br. Testamos na FactPolCheckBr de **duas formas**: com o **texto da checagem** (escrito pela agência) e com o **boato extraído** (como circulou).

**Como ler:** se o recall no texto da agência for **muito maior** que no boato, o modelo reconhece o **estilo da agência**. Isso não prova detecção de desinformação.

In [ ]:
# Texto da checagem (agência) dos mesmos 539 itens
destino = os.path.join(BRUTOS, 'Dataset-FactPolCheckBr')
if not os.path.isdir(destino):
    os.makedirs(BRUTOS, exist_ok=True)
    subprocess.run(['git', 'clone', '--depth', '1', '-q', 'https://github.com/Interfaces-UFSCAR/Dataset-FactPolCheckBr', destino], check=True)

art = pd.read_csv(os.path.join(destino, 'dados', 'com_texto.csv'))[['Link', 'texto']].rename(columns={'Link': 'link', 'texto': 'texto_checagem'})
art = art.drop_duplicates('link')
fin = pd.read_csv(os.path.join(PROC, 'fpc_eleitoral_final.csv'))[['id', 'link']]
ag = fpc[['id', 'agencia']].merge(fin, on='id').merge(art, on='link', how='left')
print('Itens sem texto da checagem:', int(ag.texto_checagem.isna().sum()))
ag = ag.dropna(subset=['texto_checagem']).reset_index(drop=True)

import nltk
# O Colab apaga estes recursos quando a sessão acaba, então baixamos de novo (rápido)
for recurso in ['stopwords', 'rslp']:
    nltk.download(recurso, quiet=True)
from nltk.corpus import stopwords
from nltk.stem import RSLPStemmer

NEGACOES = {'não', 'nem', 'nunca', 'jamais', 'nenhum', 'nenhuma', 'sem', 'nada', 'ninguém'}
STOP = set(stopwords.words('portuguese')) - NEGACOES
STEM = RSLPStemmer()

RE_URL = re.compile(r'(https?://\S+|www\.\S+)', re.I)
RE_MENCAO = re.compile(r'@\w+')
RE_HASHTAG = re.compile(r'#\w+')
RE_MARCA = re.compile(r'\(sic\.?\)|\[sic\.?\]|\(\.\.\.\)|\[\.\.\.\]', re.I)
RE_NUM = re.compile(r'\d+([.,]\d+)*')
RE_NAOLETRA = re.compile(r'[^a-zA-ZÀ-ÿ ]')

def normalizar(texto):
    t = str(texto)
    t = unicodedata.normalize('NFC', t)
    t = RE_URL.sub(' ', t)
    t = RE_MARCA.sub(' ', t)
    t = RE_MENCAO.sub(' ', t)
    t = RE_HASHTAG.sub(' ', t)
    t = t.lower()
    t = RE_NUM.sub(' num ', t)
    t = RE_NAOLETRA.sub(' ', t)
    return re.sub(r'\s+', ' ', t).strip()

def tokenizar(texto):
    return [p for p in normalizar(texto).split() if len(p) > 1 or p == 'e']

def sem_stopwords(tokens):
    return [t for t in tokens if t not in STOP]

def preparar(texto):
    toks = sem_stopwords(tokenizar(texto))
    return ' '.join(toks), ' '.join(STEM.stem(t) for t in toks)

saida = ag['texto_checagem'].apply(preparar)
ag['texto_proc'] = [s[0] for s in saida]
ag['texto_stem'] = [s[1] for s in saida]
ag['n_palavras_orig'] = ag['texto_checagem'].str.split().str.len()
fpc_boato = fpc[fpc.id.isin(ag.id)].reset_index(drop=True)
print('Itens comparados:', len(ag), '| mediana de palavras: checagem', ag.n_palavras_orig.median(), '| boato', fpc_boato.n_palavras_orig.median())

In [ ]:
linhas_vies = []
for base_nome, d_tr in [('Fake.Br', tr), ('FakeRecogna', reco_tr)]:
    for modelo in CONFIG:
        variante = escolhido[modelo]
        col = COLUNA[variante]
        # mesmos parâmetros escolhidos na busca do Fake.Br; só os dados de treino mudam
        m = clone(buscas[(modelo, variante)].best_estimator_)
        m.fit(d_tr[col], d_tr['rotulo'])
        r_boato = m.predict(fpc_boato[col]).mean()
        r_agencia = m.predict(ag[col]).mean()
        f_fake = f1_score(te['rotulo'], m.predict(te[col]))
        f_reco = f1_score(reco_te['rotulo'], m.predict(reco_te[col]))
        linhas_vies.append({'treino': base_nome, 'modelo': modelo, 'texto': variante,
                            'f1_teste_fakebr': round(f_fake, 3), 'f1_teste_fakerecogna': round(f_reco, 3),
                            'recall_boato_extraido': round(r_boato, 3), 'recall_texto_agencia': round(r_agencia, 3),
                            'diferenca_agencia_menos_boato': round(r_agencia - r_boato, 3)})
quadro_vies = pd.DataFrame(linhas_vies)
quadro_vies

## 7. Interpretação: palavras mais pesadas e erros

Atenção: palavras pesadas mostram **o que o modelo usa**, não o que é verdade. Se aparecerem nomes de fonte, de site ou marcas de formatação, é sinal de atalho.

In [ ]:
melhor_modelo = quadro_te.sort_values('f1_falso', ascending=False).iloc[0]
print('Melhor no teste interno (só para descrição):', melhor_modelo['modelo'], melhor_modelo['texto'], round(melhor_modelo['f1_falso'], 4))

def top_palavras(modelo, variante, k=25):
    est = buscas[(modelo, variante)].best_estimator_
    nomes = np.array(est.named_steps['tfidf'].get_feature_names_out())
    clf = est.named_steps['clf']
    if modelo == 'SVM':
        peso = clf.coef_[0]
    elif modelo == 'Naive Bayes':
        peso = clf.feature_log_prob_[1] - clf.feature_log_prob_[0]
    else:
        peso = clf.feature_importances_
    ordem = np.argsort(peso)
    if modelo == 'XGBoost':
        return pd.DataFrame({'palavra': nomes[ordem[::-1][:k]], 'importancia': np.round(peso[ordem[::-1][:k]], 4)})
    return pd.DataFrame({'palavras_falso': nomes[ordem[::-1][:k]], 'palavras_verdadeiro': nomes[ordem[:k]]})

tops = {m: top_palavras(m, escolhido[m]) for m in CONFIG}
for m, t in tops.items():
    print('\n==', m, '(' + escolhido[m] + ')')
    display(t) if 'display' in globals() else print(t)

In [ ]:
# Boatos que o melhor modelo NÃO reconheceu (para ler e entender os erros)
mm, vv = melhor_modelo['modelo'], melhor_modelo['texto']
fpc_err = fpc.copy()
fpc_err['previsto_falso'] = pred_fpc[(mm, vv)]
nao_reconhecidos = fpc_err[fpc_err.previsto_falso == 0][['id', 'agencia', 'texto']]
print(f'{len(nao_reconhecidos)} de {len(fpc_err)} boatos não reconhecidos por {mm} ({vv})')
nao_reconhecidos.head(15)

## 8. Salvando os resultados

In [ ]:
quadro_cv.to_csv(os.path.join(RES, 'quadro3_validacao_cruzada.csv'), index=False)
quadro_te.to_csv(os.path.join(RES, 'quadro4_teste_interno.csv'), index=False)
quadro_mc.to_csv(os.path.join(RES, 'quadro5_mcnemar.csv'), index=False)
quadro_ext.to_csv(os.path.join(RES, 'quadro6_teste_externo.csv'), index=False)
por_agencia.to_csv(os.path.join(RES, 'quadro6b_recall_por_agencia.csv'))
quadro_vies.to_csv(os.path.join(RES, 'quadro7_vies_agencia.csv'), index=False)
nao_reconhecidos.to_csv(os.path.join(RES, 'boatos_nao_reconhecidos.csv'), index=False)
for m, t in tops.items():
    t.to_csv(os.path.join(RES, f'top_palavras_{m.replace(" ", "_")}.csv'), index=False)
with open(os.path.join(RES, 'melhores_parametros.json'), 'w') as f:
    json.dump({f'{m}|{v}': {k: str(x) for k, x in gs.best_params_.items()} for (m, v), gs in buscas.items()}, f, indent=1, ensure_ascii=False)
print('Resultados salvos em', RES)